# RAG with open-source embeddings + open-weight LLM (Google Colab)

Same persistent-Chroma RAG as `rag_pipeline_chroma_colab.ipynb`, but no OpenAI key:

- **Embeddings:** `sentence-transformers/all-MiniLM-L6-v2` (open-source, runs locally)
- **LLM:** Qwen2.5 Instruct (open weights, runs locally via Hugging Face Transformers)
  - GPU runtime → `Qwen/Qwen2.5-1.5B-Instruct`
  - CPU runtime → `Qwen/Qwen2.5-0.5B-Instruct`

The index is stored in a **separate** Chroma folder from the OpenAI notebook, because embedding dimensions are different and the two indexes cannot be mixed.

**Recommended:** Runtime → Change runtime type → T4 GPU. First run downloads the models (a few hundred MB to ~3 GB).

In [ ]:
%pip install -q pysqlite3-binary langchain-chroma chromadb langchain-huggingface langchain-core langchain-community langchain-text-splitters sentence-transformers transformers accelerate

# Colab's sqlite is often too old for Chroma; swap in a newer one before importing Chroma.
import os
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    import pysqlite3

    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except ImportError:
    pass

In [ ]:
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/langchain_rag")
    print("Persisting to Google Drive so the index survives runtime restarts.")
except Exception:
    BASE_DIR = Path("langchain_rag")
    print("Google Drive not available; persisting locally for this session only.")

DOCS_DIR = BASE_DIR / "docs"
CHROMA_DIR = BASE_DIR / "chroma_db_opensource"
DOCS_DIR.mkdir(parents=True, exist_ok=True)
print("Docs:", DOCS_DIR)
print("Chroma:", CHROMA_DIR)

In [ ]:
# Write sample docs only if they are missing, so later edits are not overwritten.
samples = {
    "crashloopbackoff.txt": (
        "Kubernetes CrashLoopBackOff\n\n"
        "A pod is stuck restarting because its container keeps crashing.\n"
        "Check: kubectl describe pod, kubectl logs, and recent events.\n"
    ),
    "imagepullbackoff.txt": (
        "Kubernetes ImagePullBackOff\n\n"
        "The node cannot pull the container image.\n"
        "Common causes include a wrong image name, a missing tag, or registry auth.\n"
        "Check: kubectl describe pod and the image pull events.\n"
    ),
}

for name, text in samples.items():
    path = DOCS_DIR / name
    if path.exists():
        print(f"Keeping existing {name}")
    else:
        path.write_text(text, encoding="utf-8")
        print(f"Wrote {name}")

In [ ]:
import hashlib

import torch
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "k8s_docs_minilm"


def file_id(path: str) -> str:
    return Path(path).name


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


# 1. Load documents
loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
docs = loader.load()
print(f"Loaded {len(docs)} document(s)")

current_hashes = {}
for doc in docs:
    doc_id = file_id(doc.metadata["source"])
    doc_hash = sha256_text(doc.page_content)
    doc.metadata["doc_id"] = doc_id
    doc.metadata["doc_hash"] = doc_hash
    current_hashes[doc_id] = doc_hash

# 2. Split into chunks (cheap; embeddings run locally only for changed files)
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunk(s)")

# 3. Open-source embeddings + persisted Chroma
embed_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading embedding model {EMBED_MODEL} on {embed_device}")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": embed_device},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

stored = vectorstore.get(include=["metadatas"])
stored_hashes = {}
for meta in stored.get("metadatas") or []:
    if meta and meta.get("doc_id") and meta.get("doc_hash"):
        stored_hashes[meta["doc_id"]] = meta["doc_hash"]

# 4. Delete chunks for files that were removed or whose content changed
to_delete = [
    doc_id
    for doc_id, old_hash in stored_hashes.items()
    if current_hashes.get(doc_id) != old_hash
]
for doc_id in to_delete:
    vectorstore.delete(where={"doc_id": doc_id})
    reason = "changed" if doc_id in current_hashes else "deleted"
    print(f"Removed old chunks for {doc_id} ({reason})")

# 5. Embed only new or changed files
to_embed = {
    doc_id for doc_id, doc_hash in current_hashes.items() if stored_hashes.get(doc_id) != doc_hash
}
chunks_to_add = [chunk for chunk in chunks if chunk.metadata["doc_id"] in to_embed]

if chunks_to_add:
    vectorstore.add_documents(chunks_to_add)
    print(
        f"Embedded {len(chunks_to_add)} chunk(s) from {len(to_embed)} file(s): "
        + ", ".join(sorted(to_embed))
    )
else:
    print("No document changes; reused the persisted Chroma index (no embedding calls).")

print(f"Index now has {len(vectorstore.get()['ids'])} chunk(s)")

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

# Smaller model on CPU so Colab does not run out of RAM.
llm_id = (
    "Qwen/Qwen2.5-1.5B-Instruct"
    if torch.cuda.is_available()
    else "Qwen/Qwen2.5-0.5B-Instruct"
)
print(f"Loading open-weight LLM: {llm_id}")

hf_llm = HuggingFacePipeline.from_model_id(
    model_id=llm_id,
    task="text-generation",
    device_map="auto",
    model_kwargs={
        "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
    },
    pipeline_kwargs={
        "max_new_tokens": 256,
        "temperature": 0.1,
        "do_sample": True,
        "return_full_text": False,
    },
)
llm = ChatHuggingFace(llm=hf_llm)
print("LLM ready.")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# 6. Retrieve the most relevant chunks for a question
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})


def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)


# 7. Prompt the model to answer only from retrieved context
prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below. "
    "If the context does not contain the answer, say you do not know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

parser = StrOutputParser()

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

In [ ]:
# 8. Ask a question
question = "Why is my pod in CrashLoopBackOff and how do I debug it?"
print(f"Question: {question}\n")

retrieved = retriever.invoke(question)
print("Retrieved chunks:")
for i, chunk in enumerate(retrieved):
    print(f"--- chunk {i} ({chunk.metadata.get('doc_id', 'unknown')}) ---")
    print(chunk.page_content)

print("\nAnswer:")
print(rag_chain.invoke(question))

### Try persistence and change detection

1. **Persistence:** run the Chroma sync cell again without changing files. You should see `No document changes; reused the persisted Chroma index`.
2. **Re-embed only what changed:** run the next cell, then re-run the sync cell. Only `crashloopbackoff.txt` should be embedded again.

The LLM stays in RAM after the load cell; you do not need to reload it unless the runtime restarts.

In [ ]:
path = DOCS_DIR / "crashloopbackoff.txt"
path.write_text(
    path.read_text(encoding="utf-8")
    + "\nAlso check: kubectl get events -n <namespace>.\n",
    encoding="utf-8",
)
print("Updated", path)
print("Re-run the Chroma sync cell next. Only this file should be re-embedded.")